# Facecat Kronos — Walk-Forward Backtest (Colab GPU)

Run the full 4-layer screener backtest with **GPU-accelerated XGBoost** on Colab.

**Prerequisites:**
- Upload `data/ohlcv_all_a.pkl` and `data/benchmark_000905.pkl` to your Google Drive under `MyDrive/kronos/data/`.
- Or adjust the paths in the Config cell below.

**Runtime:** Select *Runtime → Change runtime type → T4 GPU*.

In [ ]:
# Cell 1: Setup — install dependencies and mount Drive
!pip install -q xgboost baostock pandas numpy scipy scikit-learn
!pip install -q stable-baselines3 gymnasium
!pip install -q sb3-contrib

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/kronos'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# Cell 2: Clone repo and add to path
import sys

REPO_DIR = '/content/facecat-kronos'

if not os.path.exists(REPO_DIR):
    # Option A: Clone from GitHub (update URL to your repo)
    # !git clone https://github.com/youruser/facecat-kronos.git {REPO_DIR}

    # Option B: Copy from Drive
    !cp -r /content/drive/MyDrive/kronos/facecat-kronos {REPO_DIR}

sys.path.insert(0, REPO_DIR)
print(f'Repo at: {REPO_DIR}')
print(f'sys.path[0]: {sys.path[0]}')

In [ ]:
# Cell 3: Config — GPU XGBoost override
from screener.config import ScreenerConfig

cfg = ScreenerConfig(
    ohlcv_pickle_path=os.path.join(DRIVE_ROOT, 'data/ohlcv_all_a.pkl'),
    benchmark_pickle_path=os.path.join(DRIVE_ROOT, 'data/benchmark_000905.pkl'),
    drive_root=os.path.join(DRIVE_ROOT, 'output/screener'),
)

# Enable GPU for XGBoost (Colab T4)
cfg.layer1_xgb_params['device'] = 'cuda'
cfg.layer2_xgb_params['device'] = 'cuda'

print('XGBoost device (Layer 1):', cfg.layer1_xgb_params.get('device'))
print('XGBoost device (Layer 2):', cfg.layer2_xgb_params.get('device'))
print(f'Train: {cfg.train_start} → {cfg.train_end}')
print(f'Backtest: {cfg.backtest_start} → {cfg.backtest_end}')
print(f'Run ID: {cfg.run_id}')
print(f'Run dir: {cfg.run_dir}')
print(f'Cache dir: {cfg.cache_dir}')

In [ ]:
# Cell 4: Run backtest
import time
from screener.backtester import WalkForwardBacktester

bt = WalkForwardBacktester(cfg)

t0 = time.time()
results = bt.run(run_kronos=False)
elapsed = time.time() - t0

print(f'\nTotal wall time: {elapsed/60:.1f} min')

In [ ]:
# Cell 5: Display results and save to Drive
import pandas as pd
from google.colab import files

# Metrics
print('=== Backtest Metrics ===')
for k, v in results['metrics'].items():
    print(f'  {k:>20}: {v:.4f}' if isinstance(v, float) else f'  {k:>20}: {v}')

# NAV curve
nav = results['nav_series']
if nav is not None and len(nav) > 0:
    nav_s = pd.Series(nav) if not isinstance(nav, pd.Series) else nav
    nav_s.plot(title='Portfolio NAV', figsize=(12, 4))

# Save results + config to run dir
bt.save_results(results)

# Also save models
bt.layer1.save()
bt.layer2.save()

print(f'\nRun folder: {cfg.run_dir}')
print('Contents:')
for f in sorted(os.listdir(cfg.run_dir)):
    full = os.path.join(cfg.run_dir, f)
    if os.path.isdir(full):
        print(f'  {f}/')
        for sf in sorted(os.listdir(full)):
            print(f'    {sf}')
    else:
        print(f'  {f}')

# Download results pickle
results_path = os.path.join(cfg.run_dir, 'backtest_results.pkl')
print(f'\nDownloading {results_path}…')
files.download(results_path)

## RL Portfolio Agent (Layer 4)

Train a MaskablePPO agent on the same walk-forward windows. Uses L1+L2 signals as observations, manages a 3-stock portfolio with discrete position sizing, open/close execution timing, and action masking for A-share constraints.

**Note:** This reuses the `bt` backtester from above (L1+L2 models and data already loaded). If you skipped Cell 4, run Cells 1-3 first, then this section will train L1+L2 internally.

In [ ]:
# Cell 6: RL Smoke Test — verify env + PPO work before full run
# Uses synthetic signals (no L1+L2 needed). Should finish in <30 seconds.

import numpy as np
import pandas as pd

# ── 1. Action table ──────────────────────────────────────────────
from screener.portfolio_env import build_action_table, PortfolioEnv
actions = build_action_table(3, 3)
assert len(actions) == 63, f"Expected 63 actions, got {len(actions)}"
for a in actions:
    assert a[0] + a[1] + a[2] <= 1.0 + 1e-9, f"Weights exceed 1: {a}"
print(f"[OK] Action table: {len(actions)} actions, all weights sum <= 1")

# ── 2. Build synthetic env ───────────────────────────────────────
# 100 trading days, 50 fake stocks with random OHLCV
np.random.seed(42)
n_days, n_stocks = 100, 50
dates = pd.bdate_range("2020-01-01", periods=n_days)

ohlcv_dict = {}
for i in range(n_stocks):
    sym = f"sh.{600000+i}"
    close = 10.0 * np.exp(np.cumsum(np.random.randn(n_days) * 0.02))
    df = pd.DataFrame({
        "open": close * (1 + np.random.randn(n_days) * 0.005),
        "high": close * (1 + abs(np.random.randn(n_days) * 0.01)),
        "low":  close * (1 - abs(np.random.randn(n_days) * 0.01)),
        "close": close,
        "volume": np.random.randint(1000, 100000, n_days).astype(float),
    }, index=dates)
    ohlcv_dict[sym] = df

symbols = list(ohlcv_dict.keys())
bench_df = pd.DataFrame({"close": np.linspace(100, 110, n_days)}, index=dates)

# Fake daily signals: random L2 scores + features
daily_signals = []
for d in dates:
    scores = pd.Series(np.random.randn(n_stocks), index=symbols)
    feats = pd.DataFrame(
        np.random.randn(n_stocks, 15),
        index=symbols,
        columns=[
            "macd", "macd_signal", "macd_hist", "rsi_14", "rsi_5",
            "ma5_slope", "ma20_slope", "ma60_slope", "bb_position",
            "volume_trend", "mom_5", "mom_10", "mom_20", "atr_14", "obv_slope",
        ],
    )
    daily_signals.append({
        "date": d,
        "l2_scores": scores.sort_values(ascending=False),
        "l2_ranking": list(scores.sort_values(ascending=False).index),
        "l2_features": feats,
    })

from screener.config import ScreenerConfig
smoke_cfg = ScreenerConfig()

# ── 3. Env smoke test: random steps ─────────────────────────────
env = PortfolioEnv(smoke_cfg, daily_signals, ohlcv_dict, bench_df, training_mode=True)
obs, info = env.reset()
assert obs.shape == (37,), f"Obs shape: {obs.shape}"
assert not np.any(np.isnan(obs)), "NaN in initial obs"

total_reward = 0.0
for step in range(n_days - 1):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    assert obs.shape == (37,), f"Step {step}: obs shape {obs.shape}"
    assert not np.any(np.isnan(obs)), f"Step {step}: NaN in obs"
    total_reward += reward
    if terminated:
        break

final_nav = env._nav
print(f"[OK] Random agent: {step+1} steps, final NAV={final_nav:,.0f} "
      f"(return={final_nav/smoke_cfg.initial_capital - 1:+.2%}), "
      f"total reward={total_reward:.2f}")
assert final_nav > 0, "NAV went to zero"

# ── 4. Action mask sanity ────────────────────────────────────────
env2 = PortfolioEnv(smoke_cfg, daily_signals, ohlcv_dict, bench_df, training_mode=False)
obs2, _ = env2.reset()
mask = env2.action_masks()
assert mask.shape == (63,) and mask.dtype == bool and mask.any()
print(f"[OK] Action masks: {mask.sum()}/63 legal, {(~mask).sum()} masked (day 0)")

# ── 5. PPO smoke test: train 1000 steps ─────────────────────────
from screener.rl_trader import RLTrader

smoke_cfg_ppo = ScreenerConfig(
    rl_total_timesteps=1000,
    rl_batch_size=32,
    rl_n_steps=128,
)
train_env = PortfolioEnv(smoke_cfg_ppo, daily_signals, ohlcv_dict, bench_df, training_mode=True)
trader = RLTrader(smoke_cfg_ppo)
model = trader.train(train_env)

# Test inference with masks
from sb3_contrib.common.wrappers import ActionMasker
from sb3_contrib.common.maskable.utils import get_action_masks

test_env = PortfolioEnv(smoke_cfg_ppo, daily_signals, ohlcv_dict, bench_df, training_mode=False)
masked_test_env = ActionMasker(test_env, lambda e: e.action_masks())
obs, _ = masked_test_env.reset()
blocked = 0
for _ in range(20):
    masks = get_action_masks(masked_test_env)
    action, _ = model.predict(obs, deterministic=True, action_masks=masks)
    obs, reward, terminated, truncated, info = masked_test_env.step(int(action))
    blocked += len(info.get("blocked_trades", []))
    if terminated:
        break
print(f"[OK] PPO inference: 20 steps, NAV={test_env._nav:,.0f}, blocked={blocked}")

# Save/load round-trip
import tempfile, os
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, "smoke_model")
    trader.save(model, path)
    loaded = trader.load(path)
    obs_test, _ = masked_test_env.reset()
    masks_test = get_action_masks(masked_test_env)
    a1, _ = model.predict(obs_test, deterministic=True, action_masks=masks_test)
    a2, _ = loaded.predict(obs_test, deterministic=True, action_masks=masks_test)
    assert a1 == a2, "Loaded model gives different action"
print("[OK] Model save/load round-trip")

print("\n=== All smoke tests passed ===")

In [ ]:
# Cell 6: Run RL backtest
import time
from screener.backtester import WalkForwardBacktester

# Reuse existing backtester if available (from Cell 4), otherwise create one
if 'bt' not in dir():
    bt = WalkForwardBacktester(cfg)

t0 = time.time()
rl_results = bt.run_rl(verbose=True)
rl_elapsed = time.time() - t0

print(f'\nRL backtest wall time: {rl_elapsed/60:.1f} min')

In [ ]:
# Cell 7: Display RL results
import matplotlib.pyplot as plt

print('=== RL Backtest Metrics ===')
for k, v in rl_results['metrics'].items():
    print(f'  {k:>20}: {v:.4f}' if isinstance(v, float) else f'  {k:>20}: {v}')

print('\n=== Per-Window Results ===')
for wr in rl_results['window_results']:
    print(f"  Window {wr['window']+1}: {wr['test_start']}→{wr['test_end']}  "
          f"return={wr['test_return']*100:+.2f}%  NAV={wr['final_nav']:,.0f}  "
          f"blocked={wr['blocked_trades']}")

# NAV curve comparison
fig, ax = plt.subplots(figsize=(14, 5))
rl_nav = rl_results['nav_series']
if len(rl_nav) > 0:
    (rl_nav / rl_nav.iloc[0]).plot(ax=ax, label='RL Agent', linewidth=1.5)

# Overlay paper trader NAV if available
if 'results' in dir() and 'nav_series' in results:
    pt_nav = results['nav_series']
    if len(pt_nav) > 0:
        (pt_nav / pt_nav.iloc[0]).plot(ax=ax, label='Paper Trader', linewidth=1, alpha=0.7)

ax.set_title('Normalised NAV: RL Agent vs Paper Trader')
ax.set_ylabel('Growth of $1')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Save RL results
import pickle
rl_path = os.path.join(cfg.run_dir, 'rl_backtest_results.pkl')
os.makedirs(os.path.dirname(rl_path), exist_ok=True)
with open(rl_path, 'wb') as f:
    pickle.dump(rl_results, f)
print(f'\nRL results saved → {rl_path}')

In [ ]:
# Cell 6: List past runs
runs_dir = os.path.join(cfg.drive_root, 'runs')
if os.path.isdir(runs_dir):
    runs = sorted(os.listdir(runs_dir), reverse=True)
    print(f'Past runs ({len(runs)}):')
    for r in runs:
        run_path = os.path.join(runs_dir, r)
        contents = os.listdir(run_path)
        print(f'  {r}  ({", ".join(sorted(contents))})')
else:
    print('No runs yet.')